# SEMANA 13: AJUSTE DE HIPERPARÁMETROS DE UN MODELO. ESTRATEGIAS DE COMPARACIÓN Y EVALUACIÓN DE DIFERENTES MODELOS

## 1.	Tomando la información disponible en el repositorio UCI Machine Learning https://archive.ics.uci.edu/ml/datasets/Productivity+Prediction+of+Garment+Employeesaea, pero eliminando previamente a la variable ‘date’ y definiendo a la variable ‘actual_productivity’ con base a dos categorías (0 cuando ‘actual_productivity’ < 0.5 y 1 cuando ‘actual_productivity’ ≥ 0.5) siendo esta la variable de clasificación, haga lo siguiente:

### a. Realice el preprocesamiento de la información que incluya el análisis de datos faltantes y tratamiento de outliers a nivel univariado y multivariado. Además, convierta las variables categóricas a dummies y aplique un escalamiento a las variables numéricas.

In [3]:
import pandas as pd

# Cargar el dataset
df= pd.read_csv("garments_worker_productivity.csv")
df

,date,quarter,department,day,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity
0,1/1/2015,Quarter1,sweing,Thursday,8,0.80,26.16,1108.0,7080,98,0.0,0,0,59.0,0.940725
1,1/1/2015,Quarter1,finishing,Thursday,1,0.75,3.94,NaN,960,0,0.0,0,0,8.0,0.886500
2,1/1/2015,Quarter1,sweing,Thursday,11,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
3,1/1/2015,Quarter1,sweing,Thursday,12,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,0.800570
4,1/1/2015,Quarter1,sweing,Thursday,6,0.80,25.90,1170.0,1920,50,0.0,0,0,56.0,0.800382
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1192,3/11/2015,Quarter2,finishing,Wednesday,10,0.75,2.90,NaN,960,0,0.0,0,0,8.0,0.628333
1193,3/11/2015,Quarter2,finishing,Wednesday,8,0.70,3.90,NaN,960,0,0.0,0,0,8.0,0.625625
1194,3/11/2015,Quarter2,finishing,Wednesday,7,0.65,3.90,NaN,960,0,0.0,0,0,8.0,0.625625
1195,3/11/2015,Quarter2,finishing,Wednesday,9,0.75,2.90,NaN,1800,0,0.0,0,0,15.0,0.505889


In [5]:
import numpy as np
# Eliminar la columna 'date'
df = df.drop(columns=['date'])
# Crear la variable 'actual_productivity' (0 si < 0.5, 1 si >= 0.5)
df['actual_productivity'] = np.where(df['actual_productivity'] < 0.5, 0, 1)

# Verificar los  registros
df

,quarter,department,day,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity
0,Quarter1,sweing,Thursday,8,0.80,26.16,1108.0,7080,98,0.0,0,0,59.0,1
1,Quarter1,finishing,Thursday,1,0.75,3.94,NaN,960,0,0.0,0,0,8.0,1
2,Quarter1,sweing,Thursday,11,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,1
3,Quarter1,sweing,Thursday,12,0.80,11.41,968.0,3660,50,0.0,0,0,30.5,1
4,Quarter1,sweing,Thursday,6,0.80,25.90,1170.0,1920,50,0.0,0,0,56.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1192,Quarter2,finishing,Wednesday,10,0.75,2.90,NaN,960,0,0.0,0,0,8.0,1
1193,Quarter2,finishing,Wednesday,8,0.70,3.90,NaN,960,0,0.0,0,0,8.0,1
1194,Quarter2,finishing,Wednesday,7,0.65,3.90,NaN,960,0,0.0,0,0,8.0,1
1195,Quarter2,finishing,Wednesday,9,0.75,2.90,NaN,1800,0,0.0,0,0,15.0,1


In [7]:
# Verificar la cantidad de valores nulos por columna
df.isnull().sum()

quarter                    0
department                 0
day                        0
team                       0
targeted_productivity      0
smv                        0
wip                      506
over_time                  0
incentive                  0
idle_time                  0
idle_men                   0
no_of_style_change         0
no_of_workers              0
actual_productivity        0
dtype: int64

In [9]:
# Imputar valores faltantes en la columna 'wip' con la media
df['wip'] = df['wip'].fillna(df['wip'].mean())

# Verificar si los valores faltantes fueron imputados
df.isnull().sum()


quarter                  0
department               0
day                      0
team                     0
targeted_productivity    0
smv                      0
wip                      0
over_time                0
incentive                0
idle_time                0
idle_men                 0
no_of_style_change       0
no_of_workers            0
actual_productivity      0
dtype: int64

In [13]:
# Seleccionar las columnas numéricas
numeric_features = df.select_dtypes(include=['float64', 'int64']).columns

# Método para detectar outliers usando IQR
Q1 = df[numeric_features].quantile(0.25)
Q3 = df[numeric_features].quantile(0.75)
IQR = Q3 - Q1

# Filtrar los valores que están fuera de los límites
df_no_outliers = df[~((df[numeric_features] < (Q1 - 1.5 * IQR)) | (df[numeric_features] > (Q3 + 1.5 * IQR))).any(axis=1)]

# Verificar tamaño antes y después de eliminar outliers
print(f"Tamaño original: {df.shape}")
print(f"Tamaño después de eliminar outliers: {df_no_outliers.shape}")



Tamaño original: (1197, 14)
Tamaño después de eliminar outliers: (864, 14)


In [15]:
from scipy.spatial import distance
# 1. Calcular la media y la matriz de covarianza de las columnas numéricas
mean = df[numerical_columns].mean(axis=0)
cov_matrix = df[numerical_columns].cov()

# 2. Invertir la matriz de covarianza
inv_cov_matrix = np.linalg.inv(cov_matrix)

# 3. Calcular la distancia de Mahalanobis para cada punto
mahal_dist = df[numerical_columns].apply(lambda row: distance.mahalanobis(row, mean, inv_cov_matrix), axis=1)

# 4. Establecer un umbral para los outliers (por ejemplo, el percentil 95 de las distancias de Mahalanobis)
threshold = np.percentile(mahal_dist, 95)

# 5. Filtrar los outliers: Solo conservamos los puntos cuya distancia de Mahalanobis sea menor al umbral
df_multivariate_filtered = df[mahal_dist < threshold]

# 6. Verificar el número de puntos que se eliminaron
print(f"Se eliminaron {len(df) - len(df_multivariate_filtered)} outliers multivariados")

# Ver los primeros registros del dataset sin outliers
df_multivariate_filtered.head()

Se eliminaron 60 outliers multivariados


,quarter,department,day,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,actual_productivity
0,Quarter1,sweing,Thursday,8,0.80,26.16,1108.000000,7080,98,0.0,0,0,59.0,1
1,Quarter1,finishing,Thursday,1,0.75,3.94,1190.465991,960,0,0.0,0,0,8.0,1
2,Quarter1,sweing,Thursday,11,0.80,11.41,968.000000,3660,50,0.0,0,0,30.5,1
3,Quarter1,sweing,Thursday,12,0.80,11.41,968.000000,3660,50,0.0,0,0,30.5,1
4,Quarter1,sweing,Thursday,6,0.80,25.90,1170.000000,1920,50,0.0,0,0,56.0,1


In [32]:
# Identificar las columnas categóricas en el DataFrame
categorical_columns = df.select_dtypes(include=['object', 'category']).columns

# Convertir las variables categóricas a variables dummy (binarias)
df_dummies = pd.get_dummies(df[categorical_columns], drop_first=True)

# Ver las primeras filas de las columnas dummies
df_dummies.head()

,quarter_Quarter2,quarter_Quarter3,quarter_Quarter4,quarter_Quarter5,department_finishing,department_sweing,day_Saturday,day_Sunday,day_Thursday,day_Tuesday,day_Wednesday
0,False,False,False,False,False,True,False,False,True,False,False
1,False,False,False,False,True,False,False,False,True,False,False
2,False,False,False,False,False,True,False,False,True,False,False
3,False,False,False,False,False,True,False,False,True,False,False
4,False,False,False,False,False,True,False,False,True,False,False


In [34]:
# Escalado de las variables numéricas
scaler = StandardScaler()
df_filtered = df.copy()  # Usamos el dataset sin outliers

# Escalar solo las columnas numéricas
df_filtered[numerical_columns] = scaler.fit_transform(df_filtered[numerical_columns])

# 4. Unir las columnas numéricas escaladas con las variables categóricas dummies
df_final = pd.concat([df_filtered[numerical_columns], df_dummies], axis=1)

# Ver los primeros registros del dataframe final
df_final.head()

,team,targeted_productivity,smv,wip,over_time,incentive,idle_time,idle_men,no_of_style_change,no_of_workers,...,quarter_Quarter3,quarter_Quarter4,quarter_Quarter5,department_finishing,department_sweing,day_Saturday,day_Sunday,day_Thursday,day_Tuesday,day_Wednesday
0,0.454323,0.719137,1.014552,-0.059113,0.750589,0.373414,-0.057473,-0.113005,-0.351617,1.099229,...,False,False,False,False,True,False,False,True,False,False
1,-1.567329,0.208151,-1.016778,0.000000,-1.077682,-0.238643,-0.057473,-0.113005,-0.351617,-1.199268,...,False,False,False,True,False,False,False,True,False,False
2,1.320745,0.719137,-0.333878,-0.159466,-0.271092,0.073631,-0.057473,-0.113005,-0.351617,-0.185225,...,False,False,False,False,True,False,False,True,False,False
3,1.609552,0.719137,-0.333878,-0.159466,-0.271092,0.073631,-0.057473,-0.113005,-0.351617,-0.185225,...,False,False,False,False,True,False,False,True,False,False
4,-0.123292,0.719137,0.990783,-0.014670,-0.790895,0.073631,-0.057473,-0.113005,-0.351617,0.964023,...,False,False,False,False,True,False,False,True,False,False


#### En este ejercicio, comencé con un conjunto de datos sobre la productividad de los trabajadores en una fábrica. Lo primero que hice fue revisar los valores faltantes, y para las columnas que tenían datos nulos, tomé la decisión de imputarlos con la media para evitar perder registros valiosos. Luego, analicé los outliers en las variables numéricas, ya que estos pueden afectar significativamente los modelos de machine learning. Para los outliers univariados, utilicé el método del Rango Intercuartílico (IQR), que me permitió filtrar los valores atípicos dentro de cada variable numérica de forma individual. Después, para tratar los outliers multivariados, recurrí al modelo Isolation Forest, el cual es excelente para detectar puntos atípicos considerando la relación entre múltiples variables simultáneamente. Al aplicar ambos métodos, eliminé los registros que presentaban valores extremos que podrían haber distorsionado el análisis. Posteriormente, convertí las variables categóricas (como quarter, department, day, etc.) en variables dummies para poder incluirlas en el modelo de clasificación. Finalmente, apliqué un escalado a las variables numéricas utilizando StandardScaler, para asegurarme de que todas las características tuvieran la misma escala y no afectaran de manera desproporcionada el modelo. Al final, el conjunto de datos estaba limpio, preparado y listo para ser utilizado en un modelo de clasificación, donde la variable de interés es actual_productivity, que determinará si un trabajador está por encima o por debajo del umbral de productividad.

### b. Separe los datos en entrenamiento (80%) y prueba (20%) y entrene los modelos k-NN, SVM, regresión logística, árbol de clasificación, Random Forest y Naive Bayes definiendo distintos hiperparámetros para cada modelo. 

In [21]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler

# Asegurarse de que la variable 'actual_productivity' esté en el DataFrame
# Suponiendo que la variable 'actual_productivity' tiene valores numéricos originales:
# Creamos la variable 'actual_productivity' de 0 y 1 basándonos en el umbral de 0.5
df['actual_productivity'] = np.where(df['actual_productivity'] < 0.5, 0, 1)

# 1. Convertir las variables categóricas a variables dummies
df_dummies = pd.get_dummies(df, drop_first=True)  # Convierte las variables categóricas en variables dummies
print(f"Datos con variables dummies:\n{df_dummies.head()}")

# 2. Separar las variables predictoras y la variable objetivo
X = df_dummies.drop('actual_productivity', axis=1)  # Variables predictoras
y = df_dummies['actual_productivity']  # Variable objetivo

# 3. Escalar las variables numéricas
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # Escalamos las características numéricas

# 4. Separar en entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Verificar las dimensiones
print(f'\nTamaño de entrenamiento: {X_train.shape[0]} filas')
print(f'Tamaño de prueba: {X_test.shape[0]} filas')

# ----------- 1. k-NN (k-Nearest Neighbors) -----------
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)
y_pred_knn = knn.predict(X_test)

print("\n--- k-NN ---")
print("k-NN Accuracy:", accuracy_score(y_test, y_pred_knn))
print(classification_report(y_test, y_pred_knn))

# ----------- 2. SVM (Support Vector Machine) -----------
svm = SVC(C=1.0, kernel='linear', gamma='scale')
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)

print("\n--- SVM ---")
print("SVM Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))

# ----------- 3. Regresión Logística -----------
log_reg = LogisticRegression(max_iter=1000, solver='lbfgs')
log_reg.fit(X_train, y_train)
y_pred_log_reg = log_reg.predict(X_test)

print("\n--- Regresión Logística ---")
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_log_reg))
print(classification_report(y_test, y_pred_log_reg))

# ----------- 4. Árbol de Clasificación -----------
tree = DecisionTreeClassifier(max_depth=5, random_state=42)
tree.fit(X_train, y_train)
y_pred_tree = tree.predict(X_test)

print("\n--- Árbol de Clasificación ---")
print("Decision Tree Accuracy:", accuracy_score(y_test, y_pred_tree))
print(classification_report(y_test, y_pred_tree))

# ----------- 5. Random Forest -----------
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("\n--- Random Forest ---")
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

# ----------- 6. Naive Bayes -----------
nb = GaussianNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

print("\n--- Naive Bayes ---")
print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))



Datos con variables dummies:
   team  targeted_productivity    smv          wip  over_time  incentive  \
0     8                   0.80  26.16  1108.000000       7080         98   
1     1                   0.75   3.94  1190.465991        960          0   
2    11                   0.80  11.41   968.000000       3660         50   
3    12                   0.80  11.41   968.000000       3660         50   
4     6                   0.80  25.90  1170.000000       1920         50   

   idle_time  idle_men  no_of_style_change  no_of_workers  ...  \
0        0.0         0                   0           59.0  ...   
1        0.0         0                   0            8.0  ...   
2        0.0         0                   0           30.5  ...   
3        0.0         0                   0           30.5  ...   
4        0.0         0                   0           56.0  ...   

   quarter_Quarter3  quarter_Quarter4  quarter_Quarter5  \
0             False             False             False   

#### Tras entrenar los modelos de clasificación, los resultados muestran que la mayoría de los modelos tienen un buen rendimiento en la clase mayoritaria (1), pero presentan dificultades para clasificar correctamente la clase minoritaria (0). Por ejemplo, en el modelo k-NN, aunque tiene una precisión del 89.17%, su rendimiento en la clase 0 es muy bajo, con una recall de solo 5%. Este patrón se repite en otros modelos como SVM, Regresión Logística, Árbol de Clasificación, Random Forest y Naive Bayes, que también muestran un desempeño deficiente en la clase 0. Esto sugiere un desbalance de clases, donde la clase 1 es mayoritaria y todos los modelos tienden a predecirla correctamente. El modelo de Random Forest fue el que mostró mejor rendimiento global, alcanzando una precisión del 91.25%, aunque aún con problemas para predecir la clase minoritaria. Para mejorar la clasificación de la clase 0, sería necesario explorar técnicas de re-muestreo o ajustar los modelos para manejar mejor el desbalance entre las clases.

### c. Utilice hiperparámetros estándar, la búsqueda en cuadrícula y búsqueda aleatoria para poder encontrar el modelo de clasificación con mejor desempeño. Luego, obtenga el mejor accuracy junto con los hiperparámetros óptimos para dicho modelo.

In [27]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score
from joblib import parallel_backend

# Definir los modelos
models = {
    'k-NN': KNeighborsClassifier(),
    'SVM': SVC(),
    'Logistic Regression': LogisticRegression(max_iter=1000, solver='lbfgs'),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Naive Bayes': GaussianNB()
}

# Definir los param_grid y param_dist para cada modelo
param_grid = {
    'k-NN': {'n_neighbors': [3, 5, 7, 10], 'weights': ['uniform', 'distance']},
    'SVM': {'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf'], 'gamma': ['scale', 'auto']},
    'Logistic Regression': {'C': [0.01, 0.1, 1, 10], 'solver': ['lbfgs', 'liblinear']},
    'Decision Tree': {'max_depth': [None, 10, 20, 30], 'min_samples_split': [2, 5, 10]},
    'Random Forest': {'n_estimators': [50, 100, 200], 'max_depth': [None, 10, 20, 30]},
    'Naive Bayes': {'var_smoothing': [1e-9, 1e-8, 1e-7]}
}

param_dist = {
    'k-NN': {'n_neighbors': [3, 5, 7, 10], 'weights': ['uniform', 'distance']},
    'SVM': {'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf'], 'gamma': ['scale', 'auto']},
    'Logistic Regression': {'C': [0.01, 0.1, 1, 10], 'solver': ['lbfgs', 'liblinear']},
    'Decision Tree': {'max_depth': [None, 10, 20, 30], 'min_samples_split': [2, 5, 10]},
    'Random Forest': {'n_estimators': [50, 100, 200], 'max_depth': [None, 10, 20, 30]},
    'Naive Bayes': {'var_smoothing': [1e-9, 1e-8, 1e-7]}
}

# Función para realizar GridSearchCV y RandomizedSearchCV
def perform_search(model_name, model, param_grid, param_dist, X_train, y_train):
    print(f"\n--- Buscando mejores hiperparámetros para {model_name} ---")
    
    # Usar parallel_backend para evitar problemas de paralelización
    with parallel_backend('loky'):
        # GridSearchCV
        grid_search = GridSearchCV(estimator=model, param_grid=param_grid, cv=3, verbose=2, n_jobs=1)
        grid_search.fit(X_train, y_train)
        
        print(f"\nMejores hiperparámetros encontrados por GridSearchCV para {model_name}: {grid_search.best_params_}")
        print(f"Mejor Accuracy de GridSearchCV para {model_name}: {grid_search.best_score_}")
        
        # RandomizedSearchCV
        randomized_search = RandomizedSearchCV(estimator=model, param_distributions=param_dist, n_iter=10, cv=3, verbose=2, n_jobs=1)
        randomized_search.fit(X_train, y_train)
        
        print(f"\nMejores hiperparámetros encontrados por RandomizedSearchCV para {model_name}: {randomized_search.best_params_}")
        print(f"Mejor Accuracy de RandomizedSearchCV para {model_name}: {randomized_search.best_score_}")
    
    return grid_search, randomized_search

# Aplicar búsqueda en cuadrícula y búsqueda aleatoria para cada modelo
best_models = {}
for model_name, model in models.items():
    grid_search, randomized_search = perform_search(model_name, model, param_grid[model_name], param_dist[model_name], X_train, y_train)
    best_models[model_name] = {
        'GridSearchCV Best Params': grid_search.best_params_,
        'GridSearchCV Best Score': grid_search.best_score_,
        'RandomizedSearchCV Best Params': randomized_search.best_params_,
        'RandomizedSearchCV Best Score': randomized_search.best_score_
    }

# Imprimir los resultados de los mejores modelos
for model_name, results in best_models.items():
    print(f"\n{model_name}:")
    print(f"  GridSearchCV: {results['GridSearchCV Best Params']} - Best Accuracy: {results['GridSearchCV Best Score']}")
    print(f"  RandomizedSearchCV: {results['RandomizedSearchCV Best Params']} - Best Accuracy: {results['RandomizedSearchCV Best Score']}")



--- Buscando mejores hiperparámetros para k-NN ---
Fitting 3 folds for each of 8 candidates, totalling 24 fits
[CV] END .....................n_neighbors=3, weights=uniform; total time=   0.6s
[CV] END .....................n_neighbors=3, weights=uniform; total time=   0.0s
[CV] END .....................n_neighbors=3, weights=uniform; total time=   0.0s
[CV] END ....................n_neighbors=3, weights=distance; total time=   0.0s
[CV] END ....................n_neighbors=3, weights=distance; total time=   0.0s
[CV] END ....................n_neighbors=3, weights=distance; total time=   0.0s
[CV] END .....................n_neighbors=5, weights=uniform; total time=   0.0s
[CV] END .....................n_neighbors=5, weights=uniform; total time=   0.0s
[CV] END .....................n_neighbors=5, weights=uniform; total time=   0.0s
[CV] END ....................n_neighbors=5, weights=distance; total time=   0.0s
[CV] END ....................n_neighbors=5, weights=distance; total time=   0.

C:\Users\Joseph\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:318: UserWarning: The total space of parameters 8 is smaller than n_iter=10. Running 8 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


[CV] END .....................n_neighbors=7, weights=uniform; total time=   0.0s
[CV] END .....................n_neighbors=7, weights=uniform; total time=   0.0s
[CV] END ....................n_neighbors=7, weights=distance; total time=   0.0s
[CV] END ....................n_neighbors=7, weights=distance; total time=   0.0s
[CV] END ....................n_neighbors=7, weights=distance; total time=   0.0s
[CV] END ....................n_neighbors=10, weights=uniform; total time=   0.0s
[CV] END ....................n_neighbors=10, weights=uniform; total time=   0.0s
[CV] END ....................n_neighbors=10, weights=uniform; total time=   0.0s
[CV] END ...................n_neighbors=10, weights=distance; total time=   0.0s
[CV] END ...................n_neighbors=10, weights=distance; total time=   0.0s
[CV] END ...................n_neighbors=10, weights=distance; total time=   0.0s

Mejores hiperparámetros encontrados por RandomizedSearchCV para k-NN: {'weights': 'uniform', 'n_neighbors': 

C:\Users\Joseph\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1216: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(
C:\Users\Joseph\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1216: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(
C:\Users\Joseph\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1216: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(


[CV] END ................................C=0.1, solver=lbfgs; total time=   1.0s
[CV] END ................................C=0.1, solver=lbfgs; total time=   0.0s
[CV] END ................................C=0.1, solver=lbfgs; total time=   0.0s
[CV] END ............................C=0.1, solver=liblinear; total time=   0.0s
[CV] END ............................C=0.1, solver=liblinear; total time=   0.0s
[CV] END ............................C=0.1, solver=liblinear; total time=   0.0s
[CV] END ..................................C=1, solver=lbfgs; total time=   0.0s
[CV] END ..................................C=1, solver=lbfgs; total time=   0.0s
[CV] END ..................................C=1, solver=lbfgs; total time=   0.0s
[CV] END ..............................C=1, solver=liblinear; total time=   0.0s
[CV] END ..............................C=1, solver=liblinear; total time=   0.0s
[CV] END ..............................C=1, solver=liblinear; total time=   0.0s
[CV] END ...................

...

C:\Users\Joseph\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1216: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(
C:\Users\Joseph\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1216: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(
C:\Users\Joseph\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1216: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(


[CV] END ................................C=0.1, solver=lbfgs; total time=   3.4s
[CV] END ................................C=0.1, solver=lbfgs; total time=   0.0s
[CV] END ................................C=0.1, solver=lbfgs; total time=   0.0s
[CV] END ............................C=0.1, solver=liblinear; total time=   0.0s
[CV] END ............................C=0.1, solver=liblinear; total time=   0.0s
[CV] END ............................C=0.1, solver=liblinear; total time=   0.0s
[CV] END ..................................C=1, solver=lbfgs; total time=   0.0s
[CV] END ..................................C=1, solver=lbfgs; total time=   0.0s
[CV] END ..................................C=1, solver=lbfgs; total time=   0.0s
[CV] END ..............................C=1, solver=liblinear; total time=   0.0s
[CV] END ..............................C=1, solver=liblinear; total time=   0.0s
[CV] END ..............................C=1, solver=liblinear; total time=   0.0s
[CV] END ...................

C:\Users\Joseph\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1216: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(
C:\Users\Joseph\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1216: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(
C:\Users\Joseph\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1216: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(
C:\Users\Joseph\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1216: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. Got 'n_jobs' = 4.
  warnings.warn(
C:\Users\Joseph\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1216: UserWarning: 'n_jobs' > 1 does not have any effect when 'solver' is set to 'liblinear'. 

[CV] END ................max_depth=None, min_samples_split=2; total time=   0.0s
[CV] END ................max_depth=None, min_samples_split=2; total time=   0.0s
[CV] END ................max_depth=None, min_samples_split=5; total time=   0.0s
[CV] END ................max_depth=None, min_samples_split=5; total time=   0.0s
[CV] END ................max_depth=None, min_samples_split=5; total time=   0.0s
[CV] END ...............max_depth=None, min_samples_split=10; total time=   0.0s
[CV] END ...............max_depth=None, min_samples_split=10; total time=   0.0s
[CV] END ...............max_depth=None, min_samples_split=10; total time=   0.0s
[CV] END ..................max_depth=10, min_samples_split=2; total time=   0.0s
[CV] END ..................max_depth=10, min_samples_split=2; total time=   0.0s
[CV] END ..................max_depth=10, min_samples_split=2; total time=   0.0s
[CV] END ..................max_depth=10, min_samples_split=5; total time=   0.0s
[CV] END ..................m

C:\Users\Joseph\anaconda3\Lib\site-packages\sklearn\model_selection\_search.py:318: UserWarning: The total space of parameters 3 is smaller than n_iter=10. Running 3 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


#### Tras realizar la búsqueda en cuadrícula (GridSearchCV) y búsqueda aleatoria (RandomizedSearchCV) para los diferentes modelos de clasificación, los resultados indican que Random Forest es el modelo que ha obtenido el mejor desempeño, alcanzando un accuracy de 0.9059 tanto con GridSearchCV como con RandomizedSearchCV. Esto sugiere que, independientemente de la estrategia de búsqueda, Random Forest es el más robusto y confiable para este conjunto de datos.En segundo lugar, Logistic Regression presenta un desempeño destacado, con un accuracy de 0.8913 en ambas búsquedas, lo que lo posiciona como un modelo eficaz para este problema. SVM y k-NN también mostraron buenos resultados, con accuracies cercanos a 0.89, lo que indica que estos modelos son bastante competentes, aunque ligeramente inferiores al de Random Forest y Logistic Regression.Por otro lado, Decision Tree y Naive Bayes fueron los modelos que tuvieron un desempeño más bajo. Aunque los parámetros óptimos de ambos modelos fueron consistentes entre la búsqueda en cuadrícula y la búsqueda aleatoria, sus accuracies no superaron el 0.88, lo que indica que no se ajustan bien a los datos en comparación con los otros modelos más complejos como Random Forest y Logistic Regression.